# SOTU Speech Analytics: Rhetoric Measurement Framework

This notebook demonstrates a reproducible workflow for analyzing State of the Union speeches across multiple years.

Rather than treating the speeches as text to summarize, the goal is to treat them as a measurable corpus. The workflow compares speeches on agenda structure, rhetorical moves, framing, coalition targeting, and topic-level patterns.

The demo uses cleaned and chunked SOTU transcripts, topic assignments, LLM-assisted rhetoric labels, and derived metrics to explore how presidential messaging changes across years.

## Why this matters

Simple keyword counts can show which words appear most often, but they miss the structure of the argument.

This workflow asks richer questions:

- Which topics dominate each speech?
- Which rhetorical moves are used most often?
- How do tone and framing vary by topic?
- Which constituencies or coalitions are addressed?
- How do these patterns change across years?

The emphasis is on building a repeatable measurement framework, not on treating the LLM as an oracle.

## Analysis plan

The workflow is organized around a few reusable stages:

1. Load cleaned speech chunks and derived topic labels.
2. Join topic assignments with LLM-assisted rhetoric labels.
3. Validate and patch missing labels where needed.
4. Map granular topics into higher-level macro topics.
5. Compare rhetorical devices, tones, and topic patterns across years.
6. Generate tables and visualizations that make the differences easier to inspect.

The supporting source code lives under:

- `src/demos/sotu-speech-analytics`
- `assets/sotu-speech-analytics`

This notebook is intended as a portfolio-facing overview of the analysis workflow.


In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np

from pathlib import Path

def find_repo_root(start: Path = Path.cwd()) -> Path:
    current = start.resolve()
    for path in [current, *current.parents]:
        if (path / ".git").exists():
            return path
    raise RuntimeError("Could not find repo root")

REPO_ROOT = find_repo_root()
BASE = REPO_ROOT / "assets" / "sotu-speech-analytics"

PATH_TOPICS = BASE / "data/derived/topics_global/global_chunk_topics.jsonl"
PATH_TOPIC_LABELS = BASE / "data/derived/topics_global/global_topic_labels_llm.json"
PATH_RHET = BASE / "data/derived/rhetoric/global_chunk_rhetoric.jsonl"

def read_jsonl(path: Path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

topics = pd.DataFrame(read_jsonl(PATH_TOPICS))
rhet = pd.DataFrame(read_jsonl(PATH_RHET))


# Treat empty strings as NaN
rhet["error"] = rhet["error"].replace("", np.nan)

# Define "good" rows
rhet["is_good"] = rhet["tone"].notna() & rhet["device"].notna() & rhet["target"].notna() & rhet["error"].isna()

# Confidence ranking
conf_rank = {"low": 0, "med": 1, "high": 2}
rhet["conf_rank"] = rhet["confidence"].map(conf_rank).fillna(-1)

# Sort so best rows come first, then drop duplicates on chunk_id
rhet_sorted = rhet.sort_values(
    by=["is_good", "conf_rank"],
    ascending=[False, False],
)

rhet_dedup = rhet_sorted.drop_duplicates(subset=["chunk_id"], keep="first").drop(columns=["is_good", "conf_rank"])

# Now merge using the deduped rhetoric
df = topics.merge(
    rhet_dedup,
    on=["chunk_id"],  # chunk_id is unique; year/index/topic are already in topics
    how="left",
    suffixes=("", "_r")
)



In [ ]:
import json
from pathlib import Path
import pandas as pd

# Load global topic labels (LLM)
PATH_TOPIC_LABELS = BASE / "data/derived/topics_global/global_topic_labels_llm.json"
topic_labels_raw = json.loads(PATH_TOPIC_LABELS.read_text(encoding="utf-8"))

topic_label_map = {}
for k, v in topic_labels_raw.items():
    try:
        tid = int(k)
    except Exception:
        tid = int(v.get("topic_id", k))
    # v might be a dict with "label"
    topic_label_map[tid] = (v.get("label") if isinstance(v, dict) else str(v)).title()

# Canonicalize df fields: keep left-side (topics) values
df = df.copy()
df["topic_label"] = df["topic_id"].map(topic_label_map)

# Drop the redundant rhs cols
df = df.drop(columns=["year_r", "chunk_index_r", "topic_id_r", "uses_guest_example_r", "guest_count_r"])

print("Rows:", len(df))
print("Missing rhetoric labels:", df["tone"].isna().sum())
df.head(3)


In [ ]:
missing = df[df["tone"].isna()][["year","chunk_id","chunk_index","topic_id","topic_label"]].copy()
missing.shape, missing.head()


In [ ]:
ALLOWED_DEVICES = {"policy_ask","credit_claim","attack","exemplar","values","threat"}
ALLOWED_TONES = {"neutral","unifying","adversarial","upbeat","grave","urgent"}
ALLOWED_TARGETS = {"the_public","institution","special_guests","domestic_opponents","foreign_adversaries","allies","unspecified"}

def validate_pred(p):
    if not p: 
        return None
    d = p.get("device")
    t = p.get("tone")
    g = p.get("target")
    if d not in ALLOWED_DEVICES: 
        return None
    if t not in ALLOWED_TONES:
        return None
    if g not in ALLOWED_TARGETS:
        return None
    return p


In [ ]:
from sotu_analytics.prompts import build_rhetoric_prompt
from sotu_analytics.models.topic_label_llm import ollama_generate_json

# Load chunk text index
def read_jsonl(path: Path):
    out = []
    for line in path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            out.append(json.loads(line))
    return out

def safe_ollama_json(model, prompt, timeout=180, num_predict=160, retries=1):
    last_err = None
    for _ in range(retries + 1):
        try:
            return ollama_generate_json(model, prompt, temperature=0.0, timeout=timeout, num_predict=num_predict)
        except (json.JSONDecodeError, ValueError) as e:
            last_err = e
            prompt = prompt + "\n\nREMINDER: Return ONE JSON object only. No double quotes inside evidence strings.\n"
    return {}


chunk_text = {}
for y in sorted(df["year"].unique()):
    rows = read_jsonl(BASE / f"data/chunks/{y}.jsonl")
    for r in rows:
        chunk_text[r["chunk_id"]] = r["text"]

patch_rows = []
model = "qwen2.5:14b-instruct-q4_K_M"

for _, row in missing.iterrows():
    year = int(row["year"])
    cid = row["chunk_id"]
    prompt = build_rhetoric_prompt(year, chunk_text[cid])
    resp = validate_pred(safe_ollama_json(model, prompt, timeout=180, num_predict=160, retries=1))
    if resp:
        patch_rows.append({
            "year": year,
            "chunk_id": cid,
            "tone": resp.get("tone"),
            "device": resp.get("device"),
            "target": resp.get("target"),
            "uses_guest_example": resp.get("uses_guest_example"),
            "confidence": resp.get("confidence"),
            "evidence": resp.get("evidence", {}),
            "model": model,
        })

patch = pd.DataFrame(patch_rows)
patch.head(), patch["tone"].isna().sum()


In [ ]:
df2 = df.merge(patch, on=["year","chunk_id"], how="left", suffixes=("", "_patch"))

for col in ["tone","device","target","uses_guest_example","confidence","evidence","model"]:
    df2[col] = df2[col].fillna(df2[f"{col}_patch"])

df2 = df2.drop(columns=[c for c in df2.columns if c.endswith("_patch")])

print("Missing after patch:", df2["tone"].isna().sum())


In [ ]:
import json
from pathlib import Path

BASE = Path("/Users/douglasdaly/Documents/GitHub/Generative-AI/assets/sotu-speech-analytics/data/derived/topics_global")

macro_map_path = BASE / "global_topic_macro_map.json"
macro_raw = json.loads(macro_map_path.read_text(encoding="utf-8"))

macro_topic_map = {int(k): v.get("macro_topic") for k, v in macro_raw.items()}
subtopic_map = {int(k): v.get("subtopic") for k, v in macro_raw.items()}
macro_conf_map = {int(k): v.get("confidence") for k, v in macro_raw.items()}

df2["macro_topic"] = df2["topic_id"].map(macro_topic_map)
df2["subtopic"] = df2["topic_id"].map(subtopic_map)
df2["macro_confidence"] = df2["topic_id"].map(macro_conf_map)

print("Missing macro_topic:", df2["macro_topic"].isna().sum())
#df2[["year","topic_id","topic_label","macro_topic","subtopic","macro_confidence","tone","device"]].head(10)
df2.groupby(by=["macro_topic","topic_label","subtopic"]).count()[['year']].reset_index().sort_values(by=["macro_topic","year"], ascending=[True, False])


In [ ]:
df2.groupby(by=["macro_topic"]).count()[['year']].reset_index().sort_values(by="year", ascending=False).rename(columns={"year": "count"})

In [ ]:
import json
import pandas as pd
from pathlib import Path
pd.options.display.max_colwidth = 750
BASE = Path("/Users/douglasdaly/Documents/GitHub/Generative-AI/assets/sotu-speech-analytics")

# df should already contain chunk_id + subtopic
target_subtopic = "Fundamental Rights"

hits = df2[df2["subtopic"].fillna("").str.lower().eq(target_subtopic.lower())].copy()
print("Rows:", len(hits), "Years:", sorted(hits["year"].unique()))

# Load chunk text for only the years we need
chunk_text_by_id = {}
for y in sorted(hits["year"].unique()):
    p = BASE / "data" / "chunks" / f"{int(y)}.jsonl"
    with p.open("r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            chunk_text_by_id[obj["chunk_id"]] = obj.get("text", "")

hits["text"] = hits["chunk_id"].map(chunk_text_by_id)

# Inspect: sorted by year, then chunk_index
cols = ["year","chunk_id","chunk_index","macro_topic","subtopic","topic_label","tone","device","target","text"]
cols = ["year","chunk_id","topic_id", "macro_topic","subtopic","tone","device","text"]
hits.sort_values(["year","chunk_index"])[cols].head(30)


In [ ]:
def top_tone_topics(data, year: int, tone: str, top_n=8, min_chunks=10):
    sub = data[(data["year"] == year) & data["tone"].notna()].copy()
    grp = sub.groupby("topic_label")["tone"].agg(
        total="count",
        tone_count=lambda s: (s == tone).sum()
    )
    grp = grp[grp["total"] >= min_chunks].copy()
    grp["pct"] = (grp["tone_count"] / grp["total"] * 100).round(1)
    return grp.sort_values("pct", ascending=False).head(top_n)

for y in sorted(df2["year"].unique()):
    print(f"\n{y} - most adversarial topics")
    display(top_tone_topics(df2, y, "adversarial", top_n=10, min_chunks=10))
    print(f"\n{y} - most unifying topics")
    display(top_tone_topics(df2, y, "unifying", top_n=10, min_chunks=10))


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

tones = (
    df2['tone'].dropna()
      .unique()
      .tolist()
)

devices = (
        df2['device'].dropna()
            .unique()
            .tolist()
)

topic_label_order = (
    df2["topic_label"]
      .value_counts()
      .index
      .tolist()
)
topic_label_order.sort()

macro_topic_order = (
    df2["macro_topic"]
      .value_counts()
      .index
    .tolist()
)
macro_topic_order

def topic_heatmap(data, year: int, topic="macro_topic", subgroup="device"):
    if subgroup not in {"tone", "device"}:
        raise ValueError("subgroup must be 'tone' or 'device'")
    elif subgroup == "tone":
        items = tones
    else:
        items = devices
    if topic not in {"macro_topic", "topic_label"}:
        raise ValueError("topic must be 'macro_topic' or 'topic_label'")
    elif topic == "macro_topic":
        topic_order = macro_topic_order
    else:
        topic_order = topic_label_order
    sub = data[(data["year"] == year) & data[subgroup].notna()].copy()
    counts = (
        sub.groupby([topic, subgroup])
           .size()
           .unstack(fill_value=0)
           .reindex(index=topic_order, columns=items, fill_value=0)
    )

    totals = counts.sum(axis=1).replace(0, np.nan)
    pct = (counts.div(totals, axis=0) * 100.0).fillna(0.0)

    # y labels show topic share within the year (% of chunks in that year)
    topic_share = (sub[topic].value_counts(normalize=True) * 100).reindex(topic_order).fillna(0.0)
    yticklabels = [f"{t} ({topic_share[t]:.1f}%)" for t in topic_order]

    subgroup_share = (sub[subgroup].value_counts(normalize=True) * 100).reindex(getattr(sub, subgroup).unique()).fillna(0.0)
    xticklabels = [f"{t} ({subgroup_share[t]:.1f}%)" for t in getattr(sub, subgroup).unique()]

    fig, ax = plt.subplots(figsize=(15, max(5, 0.35 * len(topic_order))))
    im = ax.imshow(pct.values, aspect="auto", vmin=0, vmax=100)

    ax.set_title(f"{year} {subgroup} by topic (% within topic)")
    ax.set_xlabel(subgroup.title())
    ax.set_ylabel("Topic (% of year)")
    ax.set_xticks(range(len(items)))
    ax.set_xticklabels(xticklabels, rotation=45, ha="right")
    ax.set_yticks(range(len(topic_order)))
    ax.set_yticklabels(yticklabels)

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("% within topic")
    plt.tight_layout()
    plt.show()

for y in sorted(df2["year"].unique()):
    topic_heatmap(df2, y, subgroup="tone")
    topic_heatmap(df2, y, subgroup="device")


In [ ]:
pd.options.display.max_rows = 200
#sub = data[(data["year"] == year) & data["device"].notna()].copy()
df2.groupby(by=["macro_topic","year"])["device"].value_counts(normalize=True).unstack(fill_value=0).reindex(columns=devices, fill_value=0).add_suffix("_device").merge(
    df2.groupby(by=["macro_topic","year"])["tone"].value_counts(normalize=True).unstack(fill_value=0).reindex(columns=tones, fill_value=0).add_suffix("_tone"),
    left_index=True,
    right_index=True
)

In [ ]:
df2.groupby(by="year")["tone"].value_counts(normalize=True).unstack(fill_value=0).reindex(columns=tones, fill_value=0)

In [ ]:
df2.groupby(by="year")["macro_topic"].value_counts(normalize=True).unstack(fill_value=0).reindex(columns=macro_topic_order, fill_value=0)

In [ ]:
df2.groupby(by="year")["device"].value_counts(normalize=True).unstack(fill_value=0).reindex(columns=devices, fill_value=0)

In [ ]:
df2.head()

In [ ]:
df2.groupby(by=['year','topic_label']).count()[['chunk_index']].pivot_table(index='topic_label', columns='year', values='chunk_index', fill_value=0)